In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
import warnings
warnings.simplefilter("ignore")

### Dataset Initialization

In [5]:
result_df = pd.read_csv('formula-1-world-championship-1950-2020/results.csv') # isme iska dataset hai  we have created variale and added their bdataset to them 
#stats_df = pd.read_csv('formula-1-world-championship-1950-2020/status.csv')
qualifying_df = pd.read_csv('formula-1-world-championship-1950-2020/qualifying.csv')
pit_stops_df = pd.read_csv('formula-1-world-championship-1950-2020/pit_stops.csv')
drivers_df = pd.read_csv('formula-1-world-championship-1950-2020/drivers.csv')
races_df = pd.read_csv('formula-1-world-championship-1950-2020/races.csv')
constructor_df = pd.read_csv('formula-1-world-championship-1950-2020/constructors.csv')
driver_standings_df = pd.read_csv('formula-1-world-championship-1950-2020/driver_standings.csv')
constructor_standings_df = pd.read_csv('formula-1-world-championship-1950-2020/constructor_standings.csv')
circuits_df = pd.read_csv('formula-1-world-championship-1950-2020/circuits.csv')
lap_times_df = pd.read_csv('formula-1-world-championship-1950-2020/lap_times.csv')
sprint_results_df = pd.read_csv('formula-1-world-championship-1950-2020/sprint_results.csv')

### Dataset Information

In [6]:
datasets = {
    "Results": result_df,
    "Races": races_df,
    "Drivers": drivers_df,
    "Constructors": constructor_df,
    "Qualifying": qualifying_df,
    "Pit Stops": pit_stops_df,
    "Driver Standings": driver_standings_df,
    "Constructor Standings": constructor_standings_df,
    "Circuits": circuits_df,
    "Lap Times": lap_times_df,
    "Sprint Results": sprint_results_df
}

for name, df in datasets.items():
    print("\n" + "="*70)
    print(f"DATASET: {name}")
    print("="*70)
    df.info()


DATASET: Results
<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         26759 non-null  int64  
 1   raceId           26759 non-null  int64  
 2   driverId         26759 non-null  int64  
 3   constructorId    26759 non-null  int64  
 4   number           26759 non-null  str    
 5   grid             26759 non-null  int64  
 6   position         26759 non-null  str    
 7   positionText     26759 non-null  str    
 8   positionOrder    26759 non-null  int64  
 9   points           26759 non-null  float64
 10  laps             26759 non-null  int64  
 11  time             26759 non-null  str    
 12  milliseconds     26759 non-null  str    
 13  fastestLap       26759 non-null  str    
 14  rank             26759 non-null  str    
 15  fastestLapTime   26759 non-null  str    
 16  fastestLapSpeed  26759 non-null  str    
 17  statu

# FEATURE ENGINEERING

### Target Variable Creation

In [7]:
result_df['position_num'] = pd.to_numeric(result_df['position'], errors='coerce')
result_df['podium'] = (result_df['position_num'] <= 3).astype(int) # 1 for podium finish, 0 otherwise
print(result_df['podium'].value_counts())
print(result_df['podium'].dtype)

podium
0    23363
1     3396
Name: count, dtype: int64
int64


### Feature Renaming

In [8]:
qualifying_df = qualifying_df.rename(columns={'position': 'quali_position'})
constructor_df = constructor_df.rename(columns={'nationality': 'construct_nationality'}) #team ka naam 

### Merging of Result, Races, Qualifying, Constructor, Drivers Datasets

In [9]:
df = (
    result_df
    .merge(races_df[['raceId', 'year', 'round', 'circuitId', 'name', 'date']], on='raceId', how='left')
    .merge(qualifying_df[['raceId', 'driverId', 'quali_position']], on=['raceId','driverId'], how='left')
    .merge(constructor_df[['constructorId', 'construct_nationality']], on='constructorId', how='left')
    .merge(drivers_df[['driverId', 'nationality', 'dob','forename','surname']], on='driverId', how='left')
)

In [10]:
print(list(df.columns))

['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'position_num', 'podium', 'year', 'round', 'circuitId', 'name', 'date', 'quali_position', 'construct_nationality', 'nationality', 'dob', 'forename', 'surname']


In [11]:
df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,round,circuitId,name,date,quali_position,construct_nationality,nationality,dob,forename,surname
0,1,18,1,1,22,1,1,1,1,10.0,...,1,1,Australian Grand Prix,2008-03-16,1.0,British,British,1985-01-07,Lewis,Hamilton
1,2,18,2,2,3,5,2,2,2,8.0,...,1,1,Australian Grand Prix,2008-03-16,5.0,German,German,1977-05-10,Nick,Heidfeld
2,3,18,3,3,7,7,3,3,3,6.0,...,1,1,Australian Grand Prix,2008-03-16,7.0,British,German,1985-06-27,Nico,Rosberg
3,4,18,4,4,5,11,4,4,4,5.0,...,1,1,Australian Grand Prix,2008-03-16,12.0,French,Spanish,1981-07-29,Fernando,Alonso
4,5,18,5,1,23,3,5,5,5,4.0,...,1,1,Australian Grand Prix,2008-03-16,3.0,British,Finnish,1981-10-19,Heikki,Kovalainen


### Feature Datatype Change

In [12]:
print(df['grid'].dtype)
print(df['dob'].dtype)
print(df['date'].dtype)

int64
str
str


In [13]:
df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print(df['grid'].dtype)
print(df['dob'].dtype)
print(df['date'].dtype)

int64
datetime64[us]
datetime64[us]


### New Feature Driver Age (in each Race)

In [14]:
df[['name','date']]

,name,date
0,Australian Grand Prix,2008-03-16
1,Australian Grand Prix,2008-03-16
2,Australian Grand Prix,2008-03-16
3,Australian Grand Prix,2008-03-16
4,Australian Grand Prix,2008-03-16
...,...,...
26754,Abu Dhabi Grand Prix,2024-12-08
26755,Abu Dhabi Grand Prix,2024-12-08
26756,Abu Dhabi Grand Prix,2024-12-08
26757,Abu Dhabi Grand Prix,2024-12-08


In [15]:
df['driver_age'] = (df['date'] - df['dob']).dt.days / 365.25

### New Feature Driver Name (Combination of 2 Features)

In [16]:
df['driver_name'] = df['forename']+' '+df['surname']

In [17]:
df[['date','driver_name','driver_age']].tail(20)

,date,driver_name,driver_age
26739,2024-12-08,Lando Norris,25.070500
26740,2024-12-08,Carlos Sainz,30.269678
26741,2024-12-08,Charles Leclerc,27.145791
26742,2024-12-08,Lewis Hamilton,39.917864
26743,2024-12-08,George Russell,26.811773
26744,2024-12-08,Max Verstappen,27.189596
26745,2024-12-08,Pierre Gasly,28.835044
26746,2024-12-08,Nico Hülkenberg,37.305955
26747,2024-12-08,Fernando Alonso,43.362081
26748,2024-12-08,Oscar Piastri,23.674196


In [19]:
# next merging of other datasets
#making new changes
pit_agg = pit_stops_df.groupby(['raceId','driverId']).agg(
    num_pitstops=('stop','count'),
    avg_pit_duration=('milliseconds','mean'),
    min_pit_duration=('milliseconds','min')
).reset_index()
df = df.merge(pit_agg, on=['raceId','driverId'], how='left')

In [20]:
!git add .
!git commit -m "merged pit stop dataset"
!git push origin DV

[DV 405f5b8] merged pit stop dataset
 3 files changed, 1333 insertions(+), 260 deletions(-)
 create mode 100644 .ipynb_checkpoints/f1-checkpoint.ipynb
 create mode 100644 .vscode/settings.json
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 10 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (7/7), 7.73 KiB | 7.73 MiB/s, done.
Total 7 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), done.
To https://github.com/harivarshan18/CSE572_GroupProject.git
   5ca5825..405f5b8  DV -> DV


In [23]:
# merging pitstops
lap_agg = lap_times_df.groupby(['raceId','driverId']).agg(
    avg_lap_ms=('milliseconds','mean'),
    std_lap_ms=('milliseconds','std'),
    best_lap_ms=('milliseconds','min'),
    laps_completed_lt=('lap','max')
).reset_index()
df = df.merge(lap_agg, on=['raceId','driverId'], how='left')

In [ ]:
# merging
ds = driver_standings_df.merge(races_df[['raceId','year','round']], on='raceId')
ds = ds.sort_values(['driverId','year','round'])
ds['prev_driver_points'] = ds.groupby('driverId')['points'].shift(1)
ds['prev_driver_position'] = ds.groupby('driverId')['position'].shift(1)
ds['prev_driver_wins'] = ds.groupby('driverId')['wins'].shift(1)
df = df.merge(ds[['raceId','driverId','prev_driver_points','prev_driver_position','prev_driver_wins']],
              on=['raceId','driverId'], how='left')

In [30]:
cs = constructor_standings_df.merge(races_df[['raceId','year','round']], on='raceId')
cs = cs.sort_values(['constructorId','year','round'])
cs['prev_constructor_points'] = cs.groupby('constructorId')['points'].shift(1)
cs['prev_constructor_position'] = cs.groupby('constructorId')['position'].shift(1)
df = df.merge(cs[['raceId','constructorId','prev_constructor_points','prev_constructor_position']],
              on=['raceId','constructorId'], how='left')

In [ ]:
#  Rolling driver podium rate (last 5 races) 
res_sorted = df[['driverId','raceId','year','round','podium']].sort_values(['driverId','year','round'])
res_sorted['rolling_podium_rate'] = (
    res_sorted.groupby('driverId')['podium']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
df = df.merge(res_sorted[['raceId','driverId','rolling_podium_rate']], on=['raceId','driverId'], how='left')

In [ ]:
#  Circuit familiarity (previous starts at circuit) 
df_circuit = df[['driverId','circuitId','raceId','year','round']].sort_values(['driverId','year','round'])
df_circuit['circuit_starts'] = df_circuit.groupby(['driverId','circuitId']).cumcount()
df = df.merge(df_circuit[['raceId','driverId','circuit_starts']], on=['raceId','driverId'], how='left')

In [31]:
print('Dataset shape after feature engineering:', df.shape)
df.head(3)

Dataset shape after feature engineering: (28333, 57)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,prev_driver_points_y,prev_driver_position_y,prev_driver_wins_y,prev_driver_points,prev_driver_position,prev_driver_wins,prev_constructor_points,prev_constructor_position,rolling_podium_rate,circuit_starts
0,1,18,1,1,22,1,1,1,1,10.0,...,109.0,2.0,4.0,109.0,2.0,4.0,218.0,11.0,0.4,1
1,2,18,2,2,3,5,2,2,2,8.0,...,61.0,5.0,0.0,61.0,5.0,0.0,101.0,2.0,0.0,8
2,3,18,3,3,7,7,3,3,3,6.0,...,20.0,9.0,0.0,20.0,9.0,0.0,33.0,4.0,0.0,2


In [1]:
!git status


On branch DV
Your branch is up to date with 'origin/DV'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   f1.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [2]:
!git add .
git commit -m "added laptime,driver and constructor standing,rolling driver podium rate "
git push

SyntaxError: invalid syntax (3666442560.py, line 1)